# BDA501 — IEEE-CIS Fraud Detection with Apache Spark

**Scope:** distributed ingestion, cleaning, EDA, class-imbalance treatment, chronological splitting, feature engineering, Spark MLlib Decision Tree, demo cases, and Parquet handover datasets.

**Recommended execution:** restart the kernel, then run cells from top to bottom. Do not skip a cell unless it is explicitly marked optional.

**Dataset:** IEEE-CIS Fraud Detection (`train_transaction`, `train_identity`, `test_transaction`, `test_identity`).

## Cell 1 — Optional dependency installer

Run once only when the active Python kernel does not already contain the required packages. Restart the kernel after installation.

In [14]:
# OPTIONAL: run once, then restart the kernel if packages were installed.
# This cell intentionally does NOT install pyarrow because the pipeline does not require it.

import importlib.util
import platform
import shutil
import subprocess
import sys

print("Python executable:", sys.executable)
print("Python version   :", platform.python_version())

if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        "Use a Python 3.11 or 3.12 Jupyter kernel for this Spark notebook. "
        f"Current version: {platform.python_version()}"
    )

requirements = {
    "pyspark": "pyspark==3.5.7",
    "pandas": "pandas>=2.1,<3.0",
    "numpy": "numpy>=1.26,<2.0",
    "matplotlib": "matplotlib>=3.8,<4.0",
    "IPython": "ipykernel>=6.29,<7.0",
}

installed_now = []
for module_name, requirement in requirements.items():
    if importlib.util.find_spec(module_name) is not None:
        print(f"OK: {module_name} is already installed")
        continue

    print(f"Installing {requirement} ...")
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", requirement],
        text=True,
        capture_output=True,
    )
    if completed.returncode != 0:
        print(completed.stdout[-3000:])
        print(completed.stderr[-5000:])
        raise RuntimeError(f"Installation failed: {requirement}")
    installed_now.append(requirement)

java_cmd = shutil.which("java")
if java_cmd is None:
    raise RuntimeError(
        "Java was not found. Install Java 17, restart VS Code, and run again. "
        "Windows command: winget install EclipseAdoptium.Temurin.17.JDK"
    )

java_result = subprocess.run([java_cmd, "-version"], text=True, capture_output=True)
print((java_result.stderr or java_result.stdout).splitlines()[0])

if installed_now:
    print("\nInstalled:")
    for item in installed_now:
        print(" -", item)
    print("\nRESTART THE JUPYTER KERNEL, then run the complete pipeline cell.")
else:
    print("\nAll required Python packages are already available.")


Python executable: c:\Users\dan13\AppData\Local\Programs\Python\Python311\python.exe
Python version   : 3.11.9
OK: pyspark is already installed
OK: pandas is already installed
OK: numpy is already installed
OK: matplotlib is already installed
OK: IPython is already installed
openjdk version "17.0.19" 2026-04-21

All required Python packages are already available.


## Cell 2 — Imports

Imports Python, plotting, Spark DataFrame, Spark SQL, and MLlib components used throughout the notebook.

In [15]:
from __future__ import annotations

import csv
import json
import logging
import os
import platform
import re
import shutil
import sys
import time
from functools import reduce
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T


## Cell 3 — Runtime configuration

Defines reproducibility settings, environment flags, source filenames, and logging.

In [16]:
SEED = int(os.getenv("PIPELINE_SEED", "42"))
np.random.seed(SEED)

RUN_MODEL_DEMO = os.getenv("RUN_MODEL_DEMO", "true").lower() in {"1", "true", "yes"}
WRITE_WIDE_FEATURE_STORE = os.getenv("WRITE_WIDE_FEATURE_STORE", "false").lower() in {"1", "true", "yes"}
UNDERSAMPLE_LEGIT_TO_FRAUD_RATIO = float(os.getenv("UNDERSAMPLE_RATIO", "4.0"))
PROFILE_BATCH_SIZE = int(os.getenv("PROFILE_BATCH_SIZE", "40"))
TOP_N = int(os.getenv("EDA_TOP_N", "20"))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("bda501_ieee_cis")

REQUIRED_FILES = [
    "train_transaction.csv",
    "train_identity.csv",
    "test_transaction.csv",
    "test_identity.csv",
]
OPTIONAL_FILES = ["sample_submission.csv"]


## Cell 4 — Project and dataset discovery functions

Locates the project root and the IEEE-CIS CSV directory without hard-coding only one layout.

In [17]:
def contains_required_files(directory: Path) -> bool:
    return directory.is_dir() and all((directory / name).is_file() for name in REQUIRED_FILES)


def discover_project_root() -> Path:
    configured = os.getenv("PROJECT_ROOT")
    if configured:
        return Path(configured).expanduser().resolve()

    windows_root = Path(r"D:\MSE\16. Big Data\Fraud-Detection-Score-Risk")
    if os.name == "nt" and windows_root.exists():
        return windows_root.resolve()

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / ".git").exists() or (candidate / "docker-compose.yml").exists():
            return candidate
    return cwd


def discover_raw_data_dir(project_root: Path) -> Path:
    candidates: list[Path] = []

    configured = os.getenv("IEEE_CIS_DATA_DIR")
    if configured:
        candidates.append(Path(configured).expanduser())

    candidates.extend([
        project_root / "data" / "ieee-cis-fraud-detection",
        project_root / "data" / "ieee-fraud-detection",
        project_root / "data" / "data" / "ieee-cis-fraud-detection",
        project_root / "data" / "data" / "ieee-fraud-detection",
        project_root / "data" / "raw" / "ieee-cis-fraud-detection",
        project_root / "data" / "raw" / "ieee-fraud-detection",
        project_root / "data" / "raw",
        Path("/app/data/raw"),
    ])

    if os.name == "nt":
        candidates.extend([
            Path(r"D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\ieee-cis-fraud-detection"),
            Path(r"D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection"),
        ])

    checked: set[str] = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        key = str(candidate).lower()
        if key in checked:
            continue
        checked.add(key)
        if contains_required_files(candidate):
            return candidate

    # Last-resort discovery under the project's data directory.
    data_root = project_root / "data"
    if data_root.exists():
        for train_file in data_root.rglob("train_transaction.csv"):
            parent = train_file.parent.resolve()
            if contains_required_files(parent):
                return parent

    expected = "\n".join(f" - {path}" for path in candidates[:8])
    raise FileNotFoundError(
        "Could not find the four IEEE-CIS CSV files. Checked:\n"
        f"{expected}\n\n"
        "Set IEEE_CIS_DATA_DIR to the directory containing the files if needed."
    )


## Cell 5 — Output paths and I/O helpers

Creates output folders and defines reusable JSON, CSV, and Parquet writers.

In [18]:
PROJECT_ROOT = discover_project_root()
RAW_DATA_DIR = discover_raw_data_dir(PROJECT_ROOT)
OUTPUT_DIR = Path(os.getenv(
    "IEEE_CIS_OUTPUT_DIR",
    str(PROJECT_ROOT / "data" / "processed" / "ieee_cis_spark"),
)).expanduser().resolve()

REPORTS_DIR = OUTPUT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
FEATURE_STORE_DIR = OUTPUT_DIR / "feature_store"
MODEL_READY_DIR = OUTPUT_DIR / "model_ready"
MODEL_ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
DEMO_DIR = OUTPUT_DIR / "demo"

for directory in [OUTPUT_DIR, REPORTS_DIR, FIGURES_DIR, FEATURE_STORE_DIR, MODEL_READY_DIR, MODEL_ARTIFACTS_DIR, DEMO_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def spark_path(path: Path) -> str:
    resolved = path.expanduser().resolve()
    # Spark on Windows is more reliable with a plain local path than with a file URI.
    return resolved.as_posix() if os.name == "nt" else str(resolved)


def write_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def write_small_csv(df: DataFrame, path: Path) -> None:
    """Write only small aggregate/report DataFrames as a single CSV file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.toPandas().to_csv(path, index=False)


def _fallback_write_parquet(df: DataFrame, path: Path, partition_cols: list[str] | None = None) -> None:
    import pyarrow as pa
    import pyarrow.parquet as pq

    target_dir = path.expanduser().resolve()
    if target_dir.exists():
        if target_dir.is_dir():
            shutil.rmtree(target_dir)
        else:
            target_dir.unlink()
    target_dir.mkdir(parents=True, exist_ok=True)

    partition_cols = partition_cols or []
    data_cols = [column for column in df.columns if column not in partition_cols]
    buffers: dict[tuple, list[dict]] = {}
    file_counts: dict[tuple, int] = {}
    batch_size = 50_000

    def partition_dir_name(value: object) -> str:
        if value is None:
            return "__HIVE_DEFAULT_PARTITION__"
        return re.sub(r"[^A-Za-z0-9._-]", "_", str(value))

    def flush_buffer(key: tuple) -> None:
        rows = buffers.get(key, [])
        if not rows:
            return

        if partition_cols:
            partition_path = target_dir
            for column, value in zip(partition_cols, key):
                partition_path = partition_path / f"{column}={partition_dir_name(value)}"
            partition_path.mkdir(parents=True, exist_ok=True)
            file_counts[key] = file_counts.get(key, 0) + 1
            output_file = partition_path / f"part-{file_counts[key]:05d}.parquet"
        else:
            file_counts[key] = file_counts.get(key, 0) + 1
            output_file = target_dir / f"part-{file_counts[key]:05d}.parquet"

        pq.write_table(pa.Table.from_pylist(rows), output_file)
        buffers[key] = []

    for row in df.select(*df.columns).toLocalIterator():
        row_dict = row.asDict(recursive=True)
        key = tuple(row_dict[column] for column in partition_cols) if partition_cols else tuple()
        buffers.setdefault(key, []).append({column: row_dict[column] for column in data_cols})
        if len(buffers[key]) >= batch_size:
            flush_buffer(key)

    for key in list(buffers):
        flush_buffer(key)


def write_parquet(df: DataFrame, path: Path, partition_cols: list[str] | None = None) -> None:
    writer = df.write.mode("overwrite")
    try:
        if partition_cols:
            writer.partitionBy(*partition_cols).parquet(spark_path(path))
        else:
            writer.parquet(spark_path(path))
    except Exception as exc:
        message = str(exc)
        if os.name != "nt" or ("winutils.exe" not in message and "Could not locate Hadoop executable" not in message):
            raise
        print(f"Spark parquet write failed on Windows, using pyarrow fallback for {path}: {exc}")
        _fallback_write_parquet(df, path, partition_cols)


print(json.dumps({
    "project_root": str(PROJECT_ROOT),
    "raw_data_dir": str(RAW_DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "python": sys.executable,
    "python_version": platform.python_version(),
    "seed": SEED,
}, indent=2))


{
  "project_root": "D:\\MSE\\16. Big Data\\Fraud-Detection-Score-Risk",
  "raw_data_dir": "D:\\MSE\\16. Big Data\\Fraud-Detection-Score-Risk\\data\\data\\ieee-fraud-detection",
  "output_dir": "D:\\MSE\\16. Big Data\\Fraud-Detection-Score-Risk\\data\\processed\\ieee_cis_spark",
  "python": "c:\\Users\\dan13\\AppData\\Local\\Programs\\Python\\Python311\\python.exe",
  "python_version": "3.11.9",
  "seed": 42
}


## Cell 6 — Spark session factory

Windows-safe local Spark setup. The notebook deliberately does not configure an RDD checkpoint directory.

In [19]:
def create_spark_session() -> SparkSession:
    existing = SparkSession.getActiveSession()
    if existing is not None:
        try:
            existing.stop()
        except Exception as exc:
            logger.warning("Could not stop the previous Spark session cleanly: %s", exc)

    os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

    master = os.getenv("SPARK_MASTER", "local[*]")
    cpu_count = os.cpu_count() or 4
    shuffle_partitions = int(os.getenv("SPARK_SHUFFLE_PARTITIONS", str(max(8, min(64, cpu_count * 2)))))

    builder = (
        SparkSession.builder
        .appName("BDA501-IEEE-CIS-Fraud-Pipeline")
        .master(master)
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .config("spark.sql.shuffle.partitions", str(shuffle_partitions))
        .config("spark.default.parallelism", str(max(4, cpu_count)))
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.sql.adaptive.skewJoin.enabled", "true")
        .config("spark.sql.execution.arrow.pyspark.enabled", "false")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        .config("spark.sql.parquet.compression.codec", "snappy")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.driver.memory", os.getenv("SPARK_DRIVER_MEMORY", "6g"))
        .config("spark.driver.maxResultSize", os.getenv("SPARK_DRIVER_MAX_RESULT_SIZE", "2g"))
    )

    if master.startswith("local"):
        builder = (
            builder
            .config("spark.driver.host", "127.0.0.1")
            .config("spark.driver.bindAddress", "127.0.0.1")
        )

    session = builder.getOrCreate()
    session.sparkContext.setLogLevel(os.getenv("SPARK_LOG_LEVEL", "WARN"))

    # Basic health check. No setCheckpointDir() call is used.
    health_count = session.range(0, 10, 1, 2).count()
    if health_count != 10:
        raise RuntimeError(f"Spark health check failed: expected 10 rows, received {health_count}")
    return session


## Cell 7 — Start Spark and run the health check

Creates the Spark session and displays cluster/runtime information.

In [20]:
spark = create_spark_session()
print("\nSpark version       :", spark.version)
print("Spark master        :", spark.sparkContext.master)
print("Application ID      :", spark.sparkContext.applicationId)
print("Default parallelism :", spark.sparkContext.defaultParallelism)
print("Shuffle partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", spark.sparkContext.uiWebUrl or "not available")



Spark version       : 3.5.7
Spark master        : local[*]
Application ID      : local-1784838820073
Default parallelism : 16
Shuffle partitions  : 32
Spark UI            : http://127.0.0.1:4041


## Cell 8 — Source validation and explicit schema helpers

Validates required files, reads CSV headers, builds Spark schemas, and loads CSVs using explicit types.

In [21]:
def validate_source_files(raw_dir: Path) -> dict[str, Path]:
    missing = [name for name in REQUIRED_FILES if not (raw_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing files in {raw_dir}: {missing}. "
            "The directory must contain all four IEEE-CIS transaction/identity CSV files."
        )

    result = {Path(name).stem: raw_dir / name for name in REQUIRED_FILES}
    for name in OPTIONAL_FILES:
        if (raw_dir / name).is_file():
            result[Path(name).stem] = raw_dir / name
    return result


def read_csv_header(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return next(csv.reader(handle))


def build_transaction_schema(columns: list[str]) -> T.StructType:
    categorical = {
        "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
        "addr1", "addr2", "P_emaildomain", "R_emaildomain",
        "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
    }
    fields: list[T.StructField] = []
    for name in columns:
        if name == "TransactionID":
            dtype = T.LongType()
        elif name == "isFraud":
            dtype = T.IntegerType()
        elif name == "TransactionDT":
            dtype = T.LongType()
        elif name == "TransactionAmt" or name in {"dist1", "dist2"} or re.fullmatch(r"[CDV]\d+", name):
            dtype = T.DoubleType()
        elif name in categorical:
            dtype = T.StringType()
        else:
            dtype = T.StringType()
        fields.append(T.StructField(name, dtype, True))
    return T.StructType(fields)


def build_identity_schema(columns: list[str]) -> T.StructType:
    categorical = {
        "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", "id_29",
        "id_30", "id_31", "id_33", "id_34", "id_35", "id_36", "id_37", "id_38",
        "DeviceType", "DeviceInfo",
    }
    fields: list[T.StructField] = []
    for name in columns:
        if name == "TransactionID":
            dtype = T.LongType()
        elif name in categorical:
            dtype = T.StringType()
        elif name.startswith("id_"):
            dtype = T.DoubleType()
        else:
            dtype = T.StringType()
        fields.append(T.StructField(name, dtype, True))
    return T.StructType(fields)


def read_csv_with_schema(path: Path, schema: T.StructType) -> DataFrame:
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("nullValue", "")
        .option("nanValue", "NaN")
        .option("ignoreLeadingWhiteSpace", True)
        .option("ignoreTrailingWhiteSpace", True)
        .schema(schema)
        .load(spark_path(path))
    )


## Cell 9 — Primary-key and join-audit helpers

Checks `TransactionID` quality and protects the transaction grain during identity joins.

In [22]:
def audit_key(df: DataFrame, dataset_name: str) -> dict[str, object]:
    row = df.agg(
        F.count("*").alias("rows"),
        F.count("TransactionID").alias("non_null_keys"),
        F.countDistinct("TransactionID").alias("distinct_keys"),
    ).first()
    rows = int(row["rows"])
    non_null = int(row["non_null_keys"])
    distinct_keys = int(row["distinct_keys"])
    null_keys = rows - non_null
    duplicate_non_null_keys = non_null - distinct_keys
    return {
        "dataset": dataset_name,
        "rows": rows,
        "null_transaction_ids": null_keys,
        "duplicate_non_null_transaction_ids": duplicate_non_null_keys,
        "status": "pass" if null_keys == 0 and duplicate_non_null_keys == 0 else "fail",
    }


def left_join_identity(transaction_df: DataFrame, identity_df: DataFrame, dataset_name: str) -> tuple[DataFrame, dict[str, object]]:
    identity_marker = identity_df.select("TransactionID").withColumn("__has_identity", F.lit(1))
    joined = (
        transaction_df
        .join(identity_df, on="TransactionID", how="left")
        .join(identity_marker, on="TransactionID", how="left")
        .withColumn("has_identity", F.coalesce(F.col("__has_identity"), F.lit(0)).cast("int"))
        .drop("__has_identity")
    )

    before = transaction_df.count()
    after = joined.count()
    identity_rows = identity_df.count()
    matched = joined.filter(F.col("has_identity") == 1).count()
    audit = {
        "dataset": dataset_name,
        "transaction_rows_before": before,
        "identity_rows": identity_rows,
        "joined_rows_after": after,
        "matched_identity_rows": matched,
        "unmatched_transaction_rows": after - matched,
        "row_difference": after - before,
        "status": "pass" if before == after else "fail",
    }
    return joined, audit


## Cell 10 — Dataset inventory and ≥500 MB validation

Produces the source inventory required for the BDA501 dataset-size evidence.

In [23]:
source_paths = validate_source_files(RAW_DATA_DIR)
inventory_pdf = pd.DataFrame([
    {
        "filename": path.name,
        "path": str(path),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2),
    }
    for path in source_paths.values()
]).sort_values("filename")

print("\nSOURCE INVENTORY")
print(inventory_pdf.to_string(index=False))
required_size_mb = inventory_pdf[inventory_pdf["filename"].isin(REQUIRED_FILES)]["size_mb"].sum()
print(f"Required-file total: {required_size_mb:,.2f} MB")
if required_size_mb < 500:
    raise AssertionError(
        f"BDA501 requires at least 500 MB, but the four required files total {required_size_mb:,.2f} MB."
    )
inventory_pdf.to_csv(REPORTS_DIR / "source_inventory.csv", index=False)



SOURCE INVENTORY
             filename                                                                                                path  size_mb
sample_submission.csv D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection\sample_submission.csv     5.80
    test_identity.csv     D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection\test_identity.csv    24.60
 test_transaction.csv  D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection\test_transaction.csv   584.79
   train_identity.csv    D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection\train_identity.csv    25.30
train_transaction.csv D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection\train_transaction.csv   651.69
Required-file total: 1,286.38 MB


## Cell 11 — Load IEEE-CIS CSV files with Spark

Builds schemas from headers and ingests the four required files through Spark DataFrames.

In [24]:
train_tx_schema = build_transaction_schema(read_csv_header(source_paths["train_transaction"]))
test_tx_schema = build_transaction_schema(read_csv_header(source_paths["test_transaction"]))
train_id_schema = build_identity_schema(read_csv_header(source_paths["train_identity"]))
test_id_schema = build_identity_schema(read_csv_header(source_paths["test_identity"]))

train_transaction = read_csv_with_schema(source_paths["train_transaction"], train_tx_schema)
test_transaction = read_csv_with_schema(source_paths["test_transaction"], test_tx_schema)
train_identity = read_csv_with_schema(source_paths["train_identity"], train_id_schema)
test_identity = read_csv_with_schema(source_paths["test_identity"], test_id_schema)


## Cell 12 — Key validation, transaction–identity joins, and persistence

Audits IDs, joins transaction and identity datasets, verifies row preservation, and caches merged DataFrames.

In [25]:
key_audit_rows = [
    audit_key(train_transaction, "train_transaction"),
    audit_key(test_transaction, "test_transaction"),
    audit_key(train_identity, "train_identity"),
    audit_key(test_identity, "test_identity"),
]
key_audit = spark.createDataFrame(pd.DataFrame(key_audit_rows))
print("\nKEY AUDIT")
key_audit.show(truncate=False)
if key_audit.filter(F.col("status") == "fail").count() > 0:
    raise AssertionError("TransactionID validation failed. Review reports/key_audit.csv.")
write_small_csv(key_audit, REPORTS_DIR / "key_audit.csv")

train_merged, train_join_audit = left_join_identity(train_transaction, train_identity, "train")
test_merged, test_join_audit = left_join_identity(test_transaction, test_identity, "test")
join_audit = spark.createDataFrame(pd.DataFrame([train_join_audit, test_join_audit]))
print("\nJOIN AUDIT")
join_audit.show(truncate=False)
if join_audit.filter(F.col("status") == "fail").count() > 0:
    raise AssertionError("Transaction/identity join changed the transaction grain.")
write_small_csv(join_audit, REPORTS_DIR / "join_audit.csv")

train_merged = train_merged.persist(StorageLevel.MEMORY_AND_DISK)
test_merged = test_merged.persist(StorageLevel.MEMORY_AND_DISK)
train_rows = train_merged.count()
test_rows = test_merged.count()
print(f"Merged train: {train_rows:,} rows x {len(train_merged.columns)} columns")
print(f"Merged test : {test_rows:,} rows x {len(test_merged.columns)} columns")



KEY AUDIT
+-----------------+------+--------------------+----------------------------------+------+
|dataset          |rows  |null_transaction_ids|duplicate_non_null_transaction_ids|status|
+-----------------+------+--------------------+----------------------------------+------+
|train_transaction|590540|0                   |0                                 |pass  |
|test_transaction |506691|0                   |0                                 |pass  |
|train_identity   |144233|0                   |0                                 |pass  |
|test_identity    |141907|0                   |0                                 |pass  |
+-----------------+------+--------------------+----------------------------------+------+


JOIN AUDIT
+-------+-----------------------+-------------+-----------------+---------------------+--------------------------+--------------+------+
|dataset|transaction_rows_before|identity_rows|joined_rows_after|matched_identity_rows|unmatched_transaction_rows|row_d

## Cell 13 — Missing-value profiling helpers

Computes distributed column-level missingness in batches to avoid oversized aggregation expressions.

In [26]:
def chunked(values: list[str], size: int) -> Iterable[list[str]]:
    for index in range(0, len(values), size):
        yield values[index:index + size]


def profile_missingness(df: DataFrame, dataset_name: str) -> DataFrame:
    row_count = df.count()
    dtype_map = dict(df.dtypes)
    output_rows: list[dict[str, object]] = []

    for batch in chunked(df.columns, PROFILE_BATCH_SIZE):
        expressions = [
            F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
            for column in batch
        ]
        result = df.agg(*expressions).first().asDict()
        for column in batch:
            null_count = int(result[column] or 0)
            output_rows.append({
                "dataset": dataset_name,
                "column_name": column,
                "spark_type": dtype_map.get(column, "unknown"),
                "row_count": row_count,
                "null_count": null_count,
                "null_pct": float(null_count / row_count * 100.0) if row_count else 0.0,
            })
    return spark.createDataFrame(pd.DataFrame(output_rows))


## Cell 14 — Missingness and class-imbalance EDA

Generates missingness reports, fraud/legitimate counts, fraud rate, and imbalance ratio.

In [30]:
# ============================================================
# WINDOWS HADOOP / WINUTILS CONFIGURATION
# Run BEFORE creating SparkSession
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

HADOOP_HOME = Path(r"C:\hadoop")
HADOOP_BIN = HADOOP_HOME / "bin"

WINUTILS_PATH = HADOOP_BIN / "winutils.exe"
HADOOP_DLL_PATH = HADOOP_BIN / "hadoop.dll"

HADOOP_BIN.mkdir(parents=True, exist_ok=True)

os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
os.environ["hadoop.home.dir"] = str(HADOOP_HOME)

current_path = os.environ.get("PATH", "")

if str(HADOOP_BIN).lower() not in current_path.lower():
    os.environ["PATH"] = str(HADOOP_BIN) + os.pathsep + current_path

print("=" * 70)
print("WINDOWS HADOOP CONFIGURATION")
print("=" * 70)
print("Python       :", sys.executable)
print("HADOOP_HOME  :", os.environ["HADOOP_HOME"])
print("Hadoop bin   :", HADOOP_BIN)
print("winutils.exe :", WINUTILS_PATH.exists())
print("hadoop.dll   :", HADOOP_DLL_PATH.exists())

missing_files = []

if not WINUTILS_PATH.exists():
    missing_files.append(str(WINUTILS_PATH))

if not HADOOP_DLL_PATH.exists():
    missing_files.append(str(HADOOP_DLL_PATH))

if missing_files:
    raise FileNotFoundError(
        "\nThiếu Hadoop Windows binaries:\n- "
        + "\n- ".join(missing_files)
        + "\n\nHãy đặt winutils.exe và hadoop.dll đúng vào "
          r"C:\hadoop\bin rồi Restart Kernel."
    )

result = subprocess.run(
    [str(WINUTILS_PATH), "ls", str(HADOOP_HOME)],
    capture_output=True,
    text=True
)

print("\nwinutils return code:", result.returncode)

if result.stdout:
    print(result.stdout)

if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "winutils.exe tồn tại nhưng không chạy được. "
        "Có thể winutils không đúng phiên bản Hadoop hoặc thiếu hadoop.dll."
    )

print("\nWindows Hadoop environment: READY")


WINDOWS HADOOP CONFIGURATION
Python       : c:\Users\dan13\AppData\Local\Programs\Python\Python311\python.exe
HADOOP_HOME  : C:\hadoop
Hadoop bin   : C:\hadoop\bin
winutils.exe : False
hadoop.dll   : False


FileNotFoundError: 
Thiếu Hadoop Windows binaries:
- C:\hadoop\bin\winutils.exe
- C:\hadoop\bin\hadoop.dll

Hãy đặt winutils.exe và hadoop.dll đúng vào C:\hadoop\bin rồi Restart Kernel.

In [29]:
train_missingness = profile_missingness(train_merged, "train")
test_missingness = profile_missingness(test_merged, "test")
missingness_report = train_missingness.unionByName(test_missingness)
print("\nTOP MISSING COLUMNS")
missingness_report.orderBy(F.desc("null_pct")).show(30, truncate=False)
write_parquet(missingness_report, REPORTS_DIR / "missingness_profile_parquet")
write_small_csv(missingness_report.orderBy(F.desc("null_pct")), REPORTS_DIR / "missingness_profile.csv")

class_distribution = (
    train_merged.groupBy("isFraud")
    .agg(F.count("*").alias("transaction_count"))
    .withColumn(
        "percentage",
        F.round(F.col("transaction_count") / F.sum("transaction_count").over(Window.partitionBy()) * 100, 6),
    )
    .orderBy("isFraud")
)
print("\nCLASS DISTRIBUTION")
class_distribution.show(truncate=False)
write_small_csv(class_distribution, REPORTS_DIR / "class_distribution.csv")

fraud_count = train_merged.filter(F.col("isFraud") == 1).count()
legit_count = train_merged.filter(F.col("isFraud") == 0).count()
imbalance_summary = {
    "fraud_count": fraud_count,
    "legitimate_count": legit_count,
    "fraud_rate": fraud_count / max(train_rows, 1),
    "legitimate_to_fraud_ratio": legit_count / max(fraud_count, 1),
}
write_json(imbalance_summary, REPORTS_DIR / "imbalance_summary.json")
print(json.dumps(imbalance_summary, indent=2))



TOP MISSING COLUMNS
+-------+-----------+----------+---------+----------+-----------------+
|dataset|column_name|spark_type|row_count|null_count|null_pct         |
+-------+-----------+----------+---------+----------+-----------------+
|train  |id_24      |double    |590540   |585793    |99.19615944728554|
|train  |id_25      |double    |590540   |585408    |99.13096487960172|
|train  |id_07      |double    |590540   |585385    |99.12707013919464|
|train  |id_08      |double    |590540   |585385    |99.12707013919464|
|train  |id_21      |double    |590540   |585381    |99.12639279303687|
|train  |id_26      |double    |590540   |585377    |99.12571544687913|
|train  |id_22      |double    |590540   |585371    |99.1246994276425 |
|train  |id_23      |string    |590540   |585371    |99.1246994276425 |
|train  |id_27      |string    |590540   |585371    |99.1246994276425 |
|test   |id-24      |string    |506691   |501951    |99.06451861193509|
|test   |id-25      |string    |506691   |5

Py4JJavaError: An error occurred while calling o16770.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: C:\hadoop\bin\winutils.exe -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:286)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:699)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)

## Cell 15 — Cleaning and base feature-engineering functions

Normalizes categorical fields and creates time, amount, missingness, email, card, and device features.

In [ ]:
CATEGORICAL_TO_NORMALIZE = [
    "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", "id_29",
    "id_30", "id_31", "id_33", "id_34", "id_35", "id_36", "id_37", "id_38",
]


def normalize_categoricals(df: DataFrame) -> DataFrame:
    result = df
    for column in CATEGORICAL_TO_NORMALIZE:
        if column in result.columns:
            cleaned = F.lower(F.trim(F.col(column).cast("string")))
            result = result.withColumn(column, F.when(cleaned == "", F.lit(None)).otherwise(cleaned))
    return result


def add_base_features(df: DataFrame) -> DataFrame:
    identity_columns = [column for column in df.columns if column.startswith("id_")]
    monitored_columns = [
        column for column in [
            "TransactionAmt", "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
            "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo",
            "dist1", "dist2", "C1", "C2", "C3", "D1", "D2", "D3",
        ] if column in df.columns
    ]

    selected_missing_expr = reduce(
        lambda left, right: left + right,
        [F.when(F.col(column).isNull(), 1).otherwise(0) for column in monitored_columns],
        F.lit(0),
    )
    identity_missing_expr = reduce(
        lambda left, right: left + right,
        [F.when(F.col(column).isNull(), 1).otherwise(0) for column in identity_columns],
        F.lit(0),
    ) if identity_columns else F.lit(0)

    card_key_columns = [column for column in ["card1", "card2", "card3", "card5"] if column in df.columns]
    card_key = F.concat_ws("|", *[F.coalesce(F.col(column).cast("string"), F.lit("__NA__")) for column in card_key_columns])

    return (
        df
        .withColumn("transaction_day", F.floor(F.col("TransactionDT") / 86400).cast("long"))
        .withColumn("transaction_week", F.floor(F.col("TransactionDT") / 604800).cast("long"))
        .withColumn("transaction_hour", F.floor((F.col("TransactionDT") % 86400) / 3600).cast("int"))
        .withColumn("transaction_period", F.concat(F.lit("week_"), F.col("transaction_week").cast("string")))
        .withColumn("log_transaction_amount", F.log1p(F.col("TransactionAmt").cast("double")))
        .withColumn("amount_decimal", (F.col("TransactionAmt") - F.floor(F.col("TransactionAmt"))).cast("double"))
        .withColumn(
            "amount_band",
            F.when(F.col("TransactionAmt") <= 10, "00_0_10")
            .when(F.col("TransactionAmt") <= 25, "01_10_25")
            .when(F.col("TransactionAmt") <= 50, "02_25_50")
            .when(F.col("TransactionAmt") <= 100, "03_50_100")
            .when(F.col("TransactionAmt") <= 250, "04_100_250")
            .when(F.col("TransactionAmt") <= 500, "05_250_500")
            .when(F.col("TransactionAmt") <= 1000, "06_500_1000")
            .otherwise("07_1000_plus"),
        )
        .withColumn("selected_missing_count", selected_missing_expr.cast("int"))
        .withColumn("selected_missing_ratio", (F.col("selected_missing_count") / F.lit(max(len(monitored_columns), 1))).cast("double"))
        .withColumn("identity_missing_count", identity_missing_expr.cast("int"))
        .withColumn("has_device_info", F.col("DeviceInfo").isNotNull().cast("int"))
        .withColumn("has_p_email", F.col("P_emaildomain").isNotNull().cast("int"))
        .withColumn("has_r_email", F.col("R_emaildomain").isNotNull().cast("int"))
        .withColumn(
            "same_email_domain",
            (
                F.col("P_emaildomain").isNotNull()
                & F.col("R_emaildomain").isNotNull()
                & (F.col("P_emaildomain") == F.col("R_emaildomain"))
            ).cast("int"),
        )
        .withColumn(
            "device_family",
            F.when(F.col("DeviceInfo").isNull(), "missing")
            .when(F.lower(F.col("DeviceInfo")).rlike("iphone|ipad|ios"), "apple")
            .when(F.lower(F.col("DeviceInfo")).rlike("android|samsung|sm-"), "android")
            .when(F.lower(F.col("DeviceInfo")).rlike("windows"), "windows")
            .when(F.lower(F.col("DeviceInfo")).rlike("mac"), "mac")
            .otherwise("other"),
        )
        .withColumn("card_entity_key", card_key)
        .withColumn("email_entity_key", F.coalesce(F.col("P_emaildomain"), F.lit("__NA__")))
        .withColumn(
            "device_entity_key",
            F.concat_ws(
                "|",
                F.coalesce(F.col("DeviceType"), F.lit("__NA__")),
                F.coalesce(F.col("DeviceInfo"), F.lit("__NA__")),
            ),
        )
    )


## Cell 16 — Apply cleaning and base features

Transforms and persists train/test datasets.

In [ ]:
clean_train = add_base_features(normalize_categoricals(train_merged)).persist(StorageLevel.MEMORY_AND_DISK)
clean_test = add_base_features(normalize_categoricals(test_merged)).persist(StorageLevel.MEMORY_AND_DISK)
clean_train.count()
clean_test.count()


## Cell 17 — Distributed Spark SQL EDA

Runs descriptive, grouped, and fraud-rate analyses using Spark SQL and saves report tables.

In [ ]:
clean_train.createOrReplaceTempView("train_clean")
eda_queries = {
    "fraud_overview": """
        SELECT COUNT(*) AS transactions,
               SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct,
               ROUND(AVG(TransactionAmt), 4) AS avg_amount,
               ROUND(percentile_approx(TransactionAmt, 0.5), 4) AS median_amount
        FROM train_clean
    """,
    "fraud_by_product": """
        SELECT COALESCE(ProductCD, '__MISSING__') AS ProductCD,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct,
               ROUND(AVG(TransactionAmt), 4) AS avg_amount
        FROM train_clean
        GROUP BY COALESCE(ProductCD, '__MISSING__')
        ORDER BY transactions DESC
    """,
    "fraud_by_card_network": """
        SELECT COALESCE(card4, '__MISSING__') AS card4,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY COALESCE(card4, '__MISSING__')
        ORDER BY transactions DESC
    """,
    "fraud_by_card_type": """
        SELECT COALESCE(card6, '__MISSING__') AS card6,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY COALESCE(card6, '__MISSING__')
        ORDER BY transactions DESC
    """,
    "fraud_by_device": """
        SELECT COALESCE(device_family, '__MISSING__') AS device_family,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY COALESCE(device_family, '__MISSING__')
        ORDER BY transactions DESC
    """,
    "fraud_by_hour": """
        SELECT transaction_hour,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY transaction_hour
        ORDER BY transaction_hour
    """,
    "fraud_by_week": """
        SELECT transaction_week,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY transaction_week
        ORDER BY transaction_week
    """,
    "fraud_by_amount_band": """
        SELECT amount_band,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct,
               ROUND(AVG(TransactionAmt), 4) AS avg_amount
        FROM train_clean
        GROUP BY amount_band
        ORDER BY amount_band
    """,
    "top_purchaser_email_domains": """
        SELECT COALESCE(P_emaildomain, '__MISSING__') AS P_emaildomain,
               COUNT(*) AS transactions,
               SUM(isFraud) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY COALESCE(P_emaildomain, '__MISSING__')
        HAVING COUNT(*) >= 500
        ORDER BY transactions DESC
        LIMIT 30
    """,
}

eda_results: dict[str, DataFrame] = {}
for report_name, query in eda_queries.items():
    report_df = spark.sql(query)
    eda_results[report_name] = report_df
    print(f"\nEDA: {report_name}")
    report_df.show(TOP_N, truncate=False)
    write_small_csv(report_df, REPORTS_DIR / f"{report_name}.csv")


## Cell 18 — EDA visualizations

Creates presentation-ready figures from small aggregated results only; the large raw dataset remains distributed in Spark.

In [ ]:
class_pd = class_distribution.toPandas()
ax = class_pd.plot.bar(x="isFraud", y="transaction_count", legend=False, title="IEEE-CIS Class Distribution")
ax.set_xlabel("isFraud")
ax.set_ylabel("Transactions")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=160)
plt.close()

product_pd = eda_results["fraud_by_product"].toPandas()
ax = product_pd.plot.bar(x="ProductCD", y="fraud_rate_pct", legend=False, title="Fraud Rate by ProductCD")
ax.set_ylabel("Fraud rate (%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fraud_rate_by_product.png", dpi=160)
plt.close()

week_pd = eda_results["fraud_by_week"].toPandas()
ax = week_pd.plot.line(x="transaction_week", y="fraud_rate_pct", legend=False, title="Fraud Rate by Transaction Week")
ax.set_ylabel("Fraud rate (%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fraud_rate_by_week.png", dpi=160)
plt.close()

missing_pd = (
    train_missingness.orderBy(F.desc("null_pct"))
    .limit(20)
    .select("column_name", "null_pct")
    .toPandas()
    .sort_values("null_pct")
)
ax = missing_pd.plot.barh(x="column_name", y="null_pct", legend=False, title="Top 20 Missing Columns — Train")
ax.set_xlabel("Missing values (%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_missing_columns.png", dpi=160)
plt.close()


## Cell 19 — Chronological train/validation/holdout split

Splits 70%/15%/15% by `TransactionDT`, fits the high-amount threshold on training only, and validates split integrity.

In [ ]:
q70, q85 = clean_train.approxQuantile("TransactionDT", [0.70, 0.85], 0.001)
train_base = clean_train.filter(F.col("TransactionDT") <= F.lit(q70))
validation_base = clean_train.filter((F.col("TransactionDT") > F.lit(q70)) & (F.col("TransactionDT") <= F.lit(q85)))
holdout_base = clean_train.filter(F.col("TransactionDT") > F.lit(q85))

high_amount_threshold = train_base.approxQuantile("TransactionAmt", [0.99], 0.001)[0]


def add_training_fitted_flags(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "high_amount_flag",
        F.when(F.col("TransactionAmt") >= F.lit(high_amount_threshold), 1).otherwise(0).cast("int"),
    )


train_base = add_training_fitted_flags(train_base)
validation_base = add_training_fitted_flags(validation_base)
holdout_base = add_training_fitted_flags(holdout_base)
test_base = add_training_fitted_flags(clean_test)


def split_summary(df: DataFrame, name: str) -> dict[str, object]:
    row = df.agg(
        F.count("*").alias("rows"),
        F.min("TransactionDT").alias("min_transaction_dt"),
        F.max("TransactionDT").alias("max_transaction_dt"),
        F.sum(F.when(F.col("isFraud") == 1, 1).otherwise(0)).alias("fraud"),
    ).first()
    rows = int(row["rows"])
    fraud = int(row["fraud"] or 0)
    return {
        "dataset": name,
        "rows": rows,
        "fraud": fraud,
        "legitimate": rows - fraud,
        "fraud_rate": fraud / max(rows, 1),
        "min_transaction_dt": int(row["min_transaction_dt"]),
        "max_transaction_dt": int(row["max_transaction_dt"]),
    }


split_rows = [
    split_summary(train_base, "train_70pct"),
    split_summary(validation_base, "validation_15pct"),
    split_summary(holdout_base, "holdout_15pct"),
]
split_report = spark.createDataFrame(pd.DataFrame(split_rows))
print("\nCHRONOLOGICAL SPLIT")
split_report.show(truncate=False)
write_small_csv(split_report, REPORTS_DIR / "chronological_split.csv")
if sum(row["rows"] for row in split_rows) != train_rows:
    raise AssertionError("Chronological split row counts do not sum to the merged training row count.")


## Cell 20 — Leakage-safe historical aggregation functions

Builds prior card/email/device activity using training-only information.

In [ ]:
def add_training_history_features(df: DataFrame) -> DataFrame:
    result = df
    definitions = [
        ("card_entity_key", "card"),
        ("email_entity_key", "email"),
        ("device_entity_key", "device"),
    ]
    for key_column, prefix in definitions:
        ordered = Window.partitionBy(key_column).orderBy("TransactionDT", "TransactionID")
        history = ordered.rowsBetween(Window.unboundedPreceding, -1)
        result = (
            result
            .withColumn(f"prior_{prefix}_transaction_count", F.count(F.lit(1)).over(history).cast("long"))
            .withColumn(f"prior_{prefix}_amount_sum", F.coalesce(F.sum("TransactionAmt").over(history), F.lit(0.0)).cast("double"))
            .withColumn(
                f"prior_{prefix}_avg_amount",
                F.when(
                    F.col(f"prior_{prefix}_transaction_count") > 0,
                    F.col(f"prior_{prefix}_amount_sum") / F.col(f"prior_{prefix}_transaction_count"),
                ).otherwise(F.lit(0.0)),
            )
        )
    card_order = Window.partitionBy("card_entity_key").orderBy("TransactionDT", "TransactionID")
    return result.withColumn(
        "time_since_previous_card_transaction",
        (F.col("TransactionDT") - F.lag("TransactionDT").over(card_order)).cast("double"),
    )


def build_history_lookups(train_df: DataFrame) -> dict[str, DataFrame]:
    lookups: dict[str, DataFrame] = {}
    for key_column, prefix in [
        ("card_entity_key", "card"),
        ("email_entity_key", "email"),
        ("device_entity_key", "device"),
    ]:
        lookups[prefix] = train_df.groupBy(key_column).agg(
            F.count("*").alias(f"prior_{prefix}_transaction_count"),
            F.sum("TransactionAmt").alias(f"prior_{prefix}_amount_sum"),
            F.avg("TransactionAmt").alias(f"prior_{prefix}_avg_amount"),
            F.max("TransactionDT").alias(f"{prefix}_last_train_transaction_dt"),
        )
    return lookups


def apply_history_lookups(df: DataFrame, lookups: dict[str, DataFrame]) -> DataFrame:
    result = df
    for key_column, prefix in [
        ("card_entity_key", "card"),
        ("email_entity_key", "email"),
        ("device_entity_key", "device"),
    ]:
        result = result.join(lookups[prefix], on=key_column, how="left")
        result = (
            result
            .withColumn(f"prior_{prefix}_transaction_count", F.coalesce(F.col(f"prior_{prefix}_transaction_count"), F.lit(0)).cast("long"))
            .withColumn(f"prior_{prefix}_amount_sum", F.coalesce(F.col(f"prior_{prefix}_amount_sum"), F.lit(0.0)).cast("double"))
            .withColumn(f"prior_{prefix}_avg_amount", F.coalesce(F.col(f"prior_{prefix}_avg_amount"), F.lit(0.0)).cast("double"))
        )

    result = result.withColumn(
        "time_since_previous_card_transaction",
        (F.col("TransactionDT") - F.col("card_last_train_transaction_dt")).cast("double"),
    )
    drop_columns = [
        "card_last_train_transaction_dt",
        "email_last_train_transaction_dt",
        "device_last_train_transaction_dt",
    ]
    return result.drop(*[column for column in drop_columns if column in result.columns])


## Cell 21 — Apply historical features

Applies training-fitted lookup tables to train, validation, holdout, and Kaggle test datasets.

In [ ]:
history_lookups = build_history_lookups(train_base)
train_features = add_training_history_features(train_base).persist(StorageLevel.MEMORY_AND_DISK)
validation_features = apply_history_lookups(validation_base, history_lookups).persist(StorageLevel.MEMORY_AND_DISK)
holdout_features = apply_history_lookups(holdout_base, history_lookups).persist(StorageLevel.MEMORY_AND_DISK)
test_features = apply_history_lookups(test_base, history_lookups).persist(StorageLevel.MEMORY_AND_DISK)

train_features.count()
validation_features.count()
holdout_features.count()
test_features.count()

if WRITE_WIDE_FEATURE_STORE:
    write_parquet(train_features, FEATURE_STORE_DIR / "train_wide", ["transaction_period"])
    write_parquet(validation_features, FEATURE_STORE_DIR / "validation_wide", ["transaction_period"])
    write_parquet(holdout_features, FEATURE_STORE_DIR / "holdout_wide", ["transaction_period"])
    write_parquet(test_features, FEATURE_STORE_DIR / "kaggle_test_wide", ["transaction_period"])


## Cell 22 — Select model features

Retains available and non-empty numeric/categorical features and defines the training data contract.

In [ ]:
NUMERIC_CANDIDATES = [
    "TransactionAmt", "log_transaction_amount", "amount_decimal", "TransactionDT",
    "transaction_day", "transaction_week", "transaction_hour",
    "selected_missing_count", "selected_missing_ratio", "identity_missing_count",
    "has_identity", "has_device_info", "has_p_email", "has_r_email", "same_email_domain",
    "high_amount_flag", "dist1", "dist2",
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13", "C14",
    "D1", "D2", "D3", "D4", "D5", "D10", "D15",
    "prior_card_transaction_count", "prior_card_amount_sum", "prior_card_avg_amount",
    "time_since_previous_card_transaction",
    "prior_email_transaction_count", "prior_email_amount_sum", "prior_email_avg_amount",
    "prior_device_transaction_count", "prior_device_amount_sum", "prior_device_avg_amount",
]
CATEGORICAL_CANDIDATES = [
    "ProductCD", "card4", "card6", "DeviceType", "device_family", "M4", "amount_band",
]

numeric_present = [column for column in NUMERIC_CANDIDATES if column in train_features.columns]
non_null_counts = train_features.agg(*[F.count(F.col(column)).alias(column) for column in numeric_present]).first().asDict()
NUMERIC_COLUMNS = [column for column in numeric_present if int(non_null_counts.get(column, 0)) > 0]
CATEGORICAL_COLUMNS = [column for column in CATEGORICAL_CANDIDATES if column in train_features.columns]

print("Numeric model features    :", len(NUMERIC_COLUMNS))
print("Categorical model features:", len(CATEGORICAL_COLUMNS))


def select_model_columns(df: DataFrame, include_label: bool) -> DataFrame:
    selected = ["TransactionID"]
    if include_label:
        selected.append("isFraud")
    selected.extend(NUMERIC_COLUMNS)
    selected.extend(CATEGORICAL_COLUMNS)

    result = df.select(*selected)
    for column in NUMERIC_COLUMNS:
        result = result.withColumn(column, F.col(column).cast("double"))
    if CATEGORICAL_COLUMNS:
        result = result.fillna("__MISSING__", subset=CATEGORICAL_COLUMNS)
    return result


## Cell 23 — Training-only median imputation

Fits Spark MLlib `Imputer` on the chronological training set and applies it unchanged to validation, holdout, and test.

In [ ]:
train_model_raw = select_model_columns(train_features, include_label=True)
validation_model_raw = select_model_columns(validation_features, include_label=True)
holdout_model_raw = select_model_columns(holdout_features, include_label=True)
test_model_raw = select_model_columns(test_features, include_label=False)

imputed_names = [f"{column}__imputed" for column in NUMERIC_COLUMNS]
imputer = Imputer(strategy="median", inputCols=NUMERIC_COLUMNS, outputCols=imputed_names)
imputer_model = imputer.fit(train_model_raw)


def apply_imputer(df: DataFrame) -> DataFrame:
    transformed = imputer_model.transform(df)
    identity_columns = ["TransactionID"] + (["isFraud"] if "isFraud" in transformed.columns else [])
    return transformed.select(
        *identity_columns,
        *[F.col(f"{column}__imputed").alias(column) for column in NUMERIC_COLUMNS],
        *CATEGORICAL_COLUMNS,
    )


train_model_ready = apply_imputer(train_model_raw).persist(StorageLevel.MEMORY_AND_DISK)
validation_model_ready = apply_imputer(validation_model_raw).persist(StorageLevel.MEMORY_AND_DISK)
holdout_model_ready = apply_imputer(holdout_model_raw).persist(StorageLevel.MEMORY_AND_DISK)
test_model_ready = apply_imputer(test_model_raw).persist(StorageLevel.MEMORY_AND_DISK)


## Cell 24 — Class-imbalance treatment

Creates class weights and a controlled undersampled training alternative. Validation and holdout are never resampled.

In [ ]:
def label_counts(df: DataFrame) -> tuple[int, int]:
    counts = {int(row["isFraud"]): int(row["count"]) for row in df.groupBy("isFraud").count().collect()}
    return counts.get(0, 0), counts.get(1, 0)


train_legit, train_fraud = label_counts(train_model_ready)
train_total = train_legit + train_fraud
legit_weight = train_total / (2.0 * max(train_legit, 1))
fraud_weight = train_total / (2.0 * max(train_fraud, 1))

train_weighted = train_model_ready.withColumn(
    "class_weight",
    F.when(F.col("isFraud") == 1, F.lit(fraud_weight)).otherwise(F.lit(legit_weight)).cast("double"),
).persist(StorageLevel.MEMORY_AND_DISK)

fraud_train = train_model_ready.filter(F.col("isFraud") == 1)
legit_train = train_model_ready.filter(F.col("isFraud") == 0)
desired_legit = min(train_legit, int(train_fraud * UNDERSAMPLE_LEGIT_TO_FRAUD_RATIO))
legit_fraction = min(1.0, desired_legit / max(train_legit, 1))
legit_sample = legit_train.sample(withReplacement=False, fraction=legit_fraction, seed=SEED)
train_balanced = fraud_train.unionByName(legit_sample).repartition(max(4, spark.sparkContext.defaultParallelism)).persist(StorageLevel.MEMORY_AND_DISK)

balanced_legit, balanced_fraud = label_counts(train_balanced)
validation_legit, validation_fraud = label_counts(validation_model_ready)
holdout_legit, holdout_fraud = label_counts(holdout_model_ready)

balance_rows = [
    {"dataset": "train_original", "legitimate": train_legit, "fraud": train_fraud, "legit_to_fraud_ratio": train_legit / max(train_fraud, 1)},
    {"dataset": "train_controlled_undersample", "legitimate": balanced_legit, "fraud": balanced_fraud, "legit_to_fraud_ratio": balanced_legit / max(balanced_fraud, 1)},
    {"dataset": "validation_untouched", "legitimate": validation_legit, "fraud": validation_fraud, "legit_to_fraud_ratio": validation_legit / max(validation_fraud, 1)},
    {"dataset": "holdout_untouched", "legitimate": holdout_legit, "fraud": holdout_fraud, "legit_to_fraud_ratio": holdout_legit / max(holdout_fraud, 1)},
]
balance_report = spark.createDataFrame(pd.DataFrame(balance_rows))
print("\nCLASS-IMBALANCE TREATMENT")
balance_report.show(truncate=False)
write_small_csv(balance_report, REPORTS_DIR / "class_balance_report.csv")


## Cell 25 — Export training-ready Parquet datasets

Writes original, weighted, balanced, validation, holdout, and Kaggle-test feature datasets.

In [ ]:
write_parquet(train_model_ready, MODEL_READY_DIR / "train_original")
write_parquet(train_weighted, MODEL_READY_DIR / "train_weighted")
write_parquet(train_balanced, MODEL_READY_DIR / "train_balanced")
write_parquet(validation_model_ready, MODEL_READY_DIR / "validation")
write_parquet(holdout_model_ready, MODEL_READY_DIR / "holdout")
write_parquet(test_model_ready, MODEL_READY_DIR / "kaggle_test")


## Cell 26 — Decision Tree evaluation helpers

Defines fraud-class precision, recall, F1, PR-AUC, ROC-AUC, and the Spark MLlib Decision Tree pipeline.

In [ ]:
def calculate_binary_metrics(predictions: DataFrame, model_name: str, dataset_name: str, training_seconds: float) -> dict[str, object]:
    counts = {
        (int(row["isFraud"]), int(row["prediction"])): int(row["count"])
        for row in predictions.groupBy("isFraud", "prediction").count().collect()
    }
    tp = counts.get((1, 1), 0)
    tn = counts.get((0, 0), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    pr_auc = BinaryClassificationEvaluator(
        labelCol="isFraud", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
    ).evaluate(predictions)
    roc_auc = BinaryClassificationEvaluator(
        labelCol="isFraud", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(predictions)
    return {
        "model": model_name,
        "dataset": dataset_name,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "precision_fraud": precision,
        "recall_fraud": recall,
        "f1_fraud": f1,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "training_seconds": training_seconds,
    }


def build_tree_pipeline(use_weight_col: bool) -> Pipeline:
    indexers = [
        StringIndexer(inputCol=column, outputCol=f"{column}__idx", handleInvalid="keep")
        for column in CATEGORICAL_COLUMNS
    ]
    assembled_inputs = NUMERIC_COLUMNS + [f"{column}__idx" for column in CATEGORICAL_COLUMNS]
    assembler = VectorAssembler(inputCols=assembled_inputs, outputCol="features", handleInvalid="keep")

    classifier = DecisionTreeClassifier(
        labelCol="isFraud",
        featuresCol="features",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        maxDepth=int(os.getenv("DT_MAX_DEPTH", "8")),
        maxBins=int(os.getenv("DT_MAX_BINS", "64")),
        minInstancesPerNode=int(os.getenv("DT_MIN_INSTANCES_PER_NODE", "50")),
        seed=SEED,
    )
    if use_weight_col and classifier.hasParam("weightCol"):
        classifier = classifier.setParams(weightCol="class_weight")
    return Pipeline(stages=[*indexers, assembler, classifier])


## Cell 27 — Train and compare Decision Tree models

Trains a baseline and an imbalance-aware model, selects the champion by validation PR-AUC, and evaluates once on holdout.

In [ ]:
model_metrics: list[dict[str, object]] = []
model_records: dict[str, dict[str, object]] = {}
champion_name: str | None = None
champion_holdout_predictions: DataFrame | None = None
champion_test_predictions: DataFrame | None = None

if RUN_MODEL_DEMO:
    training_jobs = [
        ("baseline_tree", train_model_ready, False, "original_imbalanced"),
    ]

    probe_classifier = DecisionTreeClassifier()
    if probe_classifier.hasParam("weightCol"):
        training_jobs.append(("weighted_tree", train_weighted, True, "class_weighted"))
    else:
        training_jobs.append(("undersampled_tree", train_balanced, False, "controlled_undersampling"))

    for model_name, training_df, use_weight_col, imbalance_strategy in training_jobs:
        print(f"\nTraining {model_name} using {imbalance_strategy} ...")
        pipeline = build_tree_pipeline(use_weight_col=use_weight_col)
        started = time.time()
        fitted_model = pipeline.fit(training_df)
        training_seconds = time.time() - started

        model_path = MODEL_ARTIFACTS_DIR / model_name
        if model_path.exists():
            shutil.rmtree(model_path)
        fitted_model.write().overwrite().save(spark_path(model_path))

        validation_predictions = (
            fitted_model.transform(validation_model_ready)
            .withColumn("fraud_probability", vector_to_array("probability")[1])
            .persist(StorageLevel.MEMORY_AND_DISK)
        )
        validation_predictions.count()
        validation_metrics = calculate_binary_metrics(
            validation_predictions, model_name, "validation", training_seconds
        )
        validation_metrics["imbalance_strategy"] = imbalance_strategy
        model_metrics.append(validation_metrics)
        model_records[model_name] = {
            "model": fitted_model,
            "validation_predictions": validation_predictions,
            "training_seconds": training_seconds,
            "imbalance_strategy": imbalance_strategy,
            "model_path": str(model_path),
        }

    validation_metrics_df = spark.createDataFrame(pd.DataFrame(model_metrics))
    print("\nVALIDATION MODEL COMPARISON")
    validation_metrics_df.orderBy(F.desc("pr_auc")).show(truncate=False)

    champion_name = max(model_metrics, key=lambda row: float(row["pr_auc"]))["model"]
    champion_record = model_records[champion_name]
    champion_model = champion_record["model"]

    champion_holdout_predictions = (
        champion_model.transform(holdout_model_ready)
        .withColumn("fraud_probability", vector_to_array("probability")[1])
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    champion_test_predictions = (
        champion_model.transform(test_model_ready)
        .withColumn("fraud_probability", vector_to_array("probability")[1])
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    champion_holdout_predictions.count()
    champion_test_predictions.count()

    holdout_metrics = calculate_binary_metrics(
        champion_holdout_predictions,
        champion_name,
        "holdout_final",
        float(champion_record["training_seconds"]),
    )
    holdout_metrics["imbalance_strategy"] = champion_record["imbalance_strategy"]
    model_metrics.append(holdout_metrics)

    all_metrics_df = spark.createDataFrame(pd.DataFrame(model_metrics))
    print("\nMODEL METRICS")
    all_metrics_df.show(truncate=False)
    write_small_csv(all_metrics_df, REPORTS_DIR / "decision_tree_metrics.csv")
    write_json({"champion_model": champion_name, "metrics": model_metrics}, REPORTS_DIR / "decision_tree_metrics.json")

if not RUN_MODEL_DEMO:
    logger.info("RUN_MODEL_DEMO=false: Decision Tree training was skipped.")


## Cell 28 — Feature importance and live demo cases

Exports feature importance, real fraud/legitimate transactions, confusion-matrix examples, and Kaggle test scores.

In [ ]:
if RUN_MODEL_DEMO:
# Feature importances from the champion Decision Tree stage.
    assembled_feature_names = NUMERIC_COLUMNS + [f"{column}__indexed" for column in CATEGORICAL_COLUMNS]
    tree_model = champion_model.stages[-1]
    importance_values = tree_model.featureImportances.toArray().tolist()
    importance_pdf = pd.DataFrame({
        "feature_name": assembled_feature_names,
        "importance": importance_values,
    }).sort_values("importance", ascending=False)
    importance_pdf.to_csv(REPORTS_DIR / "decision_tree_feature_importance.csv", index=False)

    # Real demo cases: actual fraud and legitimate transactions.
    demo_columns = [
        column for column in [
            "TransactionID", "isFraud", "prediction", "fraud_probability",
            "TransactionAmt", "ProductCD", "card4", "card6", "DeviceType",
            "device_family", "transaction_hour", "high_amount_flag",
            "selected_missing_ratio", "prior_card_transaction_count",
            "prior_email_transaction_count", "prior_device_transaction_count",
        ] if column in champion_holdout_predictions.columns
    ]

    actual_fraud_cases = (
        champion_holdout_predictions
        .filter(F.col("isFraud") == 1)
        .orderBy(F.desc("fraud_probability"), F.asc("TransactionID"))
        .limit(5)
        .withColumn("demo_group", F.lit("actual_fraud"))
        .select("demo_group", *demo_columns)
    )
    actual_legit_cases = (
        champion_holdout_predictions
        .filter(F.col("isFraud") == 0)
        .orderBy(F.asc("fraud_probability"), F.asc("TransactionID"))
        .limit(5)
        .withColumn("demo_group", F.lit("actual_legitimate"))
        .select("demo_group", *demo_columns)
    )
    actual_demo_cases = actual_fraud_cases.unionByName(actual_legit_cases)
    write_small_csv(actual_demo_cases, DEMO_DIR / "actual_fraud_and_legitimate_cases.csv")

    case_type = (
        F.when((F.col("isFraud") == 1) & (F.col("prediction") == 1), "true_positive")
        .when((F.col("isFraud") == 0) & (F.col("prediction") == 0), "true_negative")
        .when((F.col("isFraud") == 0) & (F.col("prediction") == 1), "false_positive")
        .otherwise("false_negative")
    )
    case_window = Window.partitionBy("case_type").orderBy(F.desc("fraud_probability"), F.asc("TransactionID"))
    confusion_demo_cases = (
        champion_holdout_predictions
        .withColumn("case_type", case_type)
        .withColumn("case_rank", F.row_number().over(case_window))
        .filter(F.col("case_rank") <= 3)
        .select("case_type", "case_rank", *demo_columns)
        .orderBy("case_type", "case_rank")
    )
    print("\nDEMO CASES")
    confusion_demo_cases.show(20, truncate=False)
    write_small_csv(confusion_demo_cases, DEMO_DIR / "confusion_matrix_demo_cases.csv")
    write_parquet(confusion_demo_cases, DEMO_DIR / "confusion_matrix_demo_cases_parquet")

    kaggle_predictions = (
        champion_test_predictions
        .select("TransactionID", F.col("fraud_probability").alias("isFraud"))
        .orderBy("TransactionID")
    )
    write_parquet(kaggle_predictions, DEMO_DIR / "kaggle_test_predictions_parquet")


## Cell 29 — Feature catalog

Documents model feature types and missing-value policies for handover.

In [ ]:
feature_catalog_rows = [
    {
        "feature_name": column,
        "feature_type": "numeric",
        "missing_value_policy": "median fitted only on chronological training split",
        "training_ready": True,
    }
    for column in NUMERIC_COLUMNS
] + [
    {
        "feature_name": column,
        "feature_type": "categorical",
        "missing_value_policy": "__MISSING__ category; indexed inside Spark ML pipeline",
        "training_ready": True,
    }
    for column in CATEGORICAL_COLUMNS
]
feature_catalog = spark.createDataFrame(pd.DataFrame(feature_catalog_rows))
write_small_csv(feature_catalog, REPORTS_DIR / "feature_catalog.csv")


## Cell 30 — Pipeline manifest

Records source evidence, Spark execution metadata, split boundaries, imbalance strategy, model results, and output locations.

In [ ]:
manifest = {
    "pipeline": "BDA501 IEEE-CIS Spark EDA and class-imbalance pipeline",
    "generated_at_utc": pd.Timestamp.utcnow().isoformat(),
    "platform": platform.platform(),
    "python_version": sys.version,
    "spark_version": spark.version,
    "spark_master": spark.sparkContext.master,
    "spark_application_id": spark.sparkContext.applicationId,
    "spark_default_parallelism": spark.sparkContext.defaultParallelism,
    "project_root": str(PROJECT_ROOT),
    "raw_data_dir": str(RAW_DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "source_files": inventory_pdf.to_dict(orient="records"),
    "source_size_required_files_mb": float(required_size_mb),
    "split_boundaries": {"q70_transaction_dt": q70, "q85_transaction_dt": q85},
    "split_report": split_rows,
    "high_amount_threshold_train_99pct": high_amount_threshold,
    "imbalance_original": imbalance_summary,
    "class_weights": {"legitimate": legit_weight, "fraud": fraud_weight},
    "undersample_target_legitimate_to_fraud_ratio": UNDERSAMPLE_LEGIT_TO_FRAUD_RATIO,
    "numeric_features": NUMERIC_COLUMNS,
    "categorical_features": CATEGORICAL_COLUMNS,
    "champion_model": champion_name,
    "model_metrics": model_metrics,
    "outputs": {
        "train_original": str(MODEL_READY_DIR / "train_original"),
        "train_weighted": str(MODEL_READY_DIR / "train_weighted"),
        "train_balanced": str(MODEL_READY_DIR / "train_balanced"),
        "validation": str(MODEL_READY_DIR / "validation"),
        "holdout": str(MODEL_READY_DIR / "holdout"),
        "kaggle_test": str(MODEL_READY_DIR / "kaggle_test"),
        "reports": str(REPORTS_DIR),
        "demo": str(DEMO_DIR),
        "models": str(MODEL_ARTIFACTS_DIR),
    },
}
write_json(manifest, OUTPUT_DIR / "manifest.json")


## Cell 31 — Handover document for Quân

Creates the training data contract and downstream usage guidance.

In [ ]:
handover_text = f"""# IEEE-CIS preprocessing handover to Quân

## Training-ready Parquet datasets

- Original chronological training set: `{MODEL_READY_DIR / 'train_original'}`
- Class-weighted training set: `{MODEL_READY_DIR / 'train_weighted'}`
- Controlled-undersampled training set: `{MODEL_READY_DIR / 'train_balanced'}`
- Untouched validation set: `{MODEL_READY_DIR / 'validation'}`
- Untouched holdout set: `{MODEL_READY_DIR / 'holdout'}`
- Kaggle test features: `{MODEL_READY_DIR / 'kaggle_test'}`

## Data contract

- Primary key: `TransactionID`
- Label: `isFraud` (training, validation and holdout only)
- Weight column: `class_weight` in `train_weighted`
- Split method: chronological 70/15/15 using `TransactionDT`
- Numeric missing values: median fitted on training only
- Categorical missing values: `__MISSING__`
- Validation and holdout class distributions are never resampled

## Recommended training input

Use `train_weighted` for models that support sample weights. Use `train_balanced` as the fallback for algorithms without weight support. Select models on validation PR-AUC and report holdout results once.

## Reports and demo

- Feature catalog: `{REPORTS_DIR / 'feature_catalog.csv'}`
- Model metrics: `{REPORTS_DIR / 'decision_tree_metrics.csv'}`
- Real fraud/legitimate demo cases: `{DEMO_DIR / 'actual_fraud_and_legitimate_cases.csv'}`
- TP/TN/FP/FN demo cases: `{DEMO_DIR / 'confusion_matrix_demo_cases.csv'}`
- Full manifest: `{OUTPUT_DIR / 'manifest.json'}`
"""
(OUTPUT_DIR / "HANDOVER_TO_QUAN.md").write_text(handover_text, encoding="utf-8")


## Cell 32 — Final output validation and completion summary

Fails clearly if any required handover artifact is missing.

In [ ]:
required_outputs = [
    MODEL_READY_DIR / "train_original",
    MODEL_READY_DIR / "train_weighted",
    MODEL_READY_DIR / "train_balanced",
    MODEL_READY_DIR / "validation",
    MODEL_READY_DIR / "holdout",
    MODEL_READY_DIR / "kaggle_test",
    REPORTS_DIR / "feature_catalog.csv",
    OUTPUT_DIR / "manifest.json",
    OUTPUT_DIR / "HANDOVER_TO_QUAN.md",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise AssertionError(f"Pipeline finished but required outputs are missing: {missing_outputs}")

print("\n" + "=" * 76)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 76)
print("Training-ready data :", MODEL_READY_DIR)
print("EDA and QA reports  :", REPORTS_DIR)
print("Demo transactions   :", DEMO_DIR)
print("Manifest            :", OUTPUT_DIR / "manifest.json")
print("Handover to Quân    :", OUTPUT_DIR / "HANDOVER_TO_QUAN.md")
print("Champion model      :", champion_name or "model demo skipped")
print("Spark master        :", spark.sparkContext.master)
print("\nFor the final cluster demonstration, set SPARK_MASTER to your Spark standalone/Docker cluster URL before running this notebook.")


## Expected outputs

```text
data/processed/ieee_cis_spark/
├── model_ready/
│   ├── train_original/
│   ├── train_weighted/
│   ├── train_balanced/
│   ├── validation/
│   ├── holdout/
│   └── kaggle_test/
├── reports/
├── figures/
├── demo/
├── artifacts/
├── manifest.json
└── HANDOVER_TO_QUAN.md
```

For the final BDA501 demonstration, show the Spark master/application ID, source-size report, Spark SQL EDA, chronological split, class-balance comparison, Decision Tree metrics, and real fraud/legitimate demo cases.